# Second retriever on a GPU (grounding plan, part 1)

Runs `rag/retrieve_v2.py` (chunks, hybrid dense + BM25 search, `bge-reranker-base`) on all
1,470 complaints and writes `passages_v2.jsonl`. On a CPU this takes hours; on a Colab T4 it
takes a few minutes.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.
The complaint text is not stored in the repository, so step 2 rebuilds it from the archived
CFPB snapshot (about 10 minutes) and checks that the complaint IDs match the team's frozen split.
The last step downloads `passages_v2.jsonl` (help-page text and scores only, no complaint text).

In [ ]:
!pip -q install datasketch faiss-cpu
import hashlib, json, os, subprocess, sys

REPO_URL = "https://github.com/parulrewatkar0692/finance-complaint-triage"
REPO_DIR = "/content/finance-complaint-triage"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
os.chdir(REPO_DIR)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

## 2. Rebuild the real complaints and check the IDs

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="Mouwiya/cfpb-consumer-complaints", repo_type="dataset",
                  revision="038f8f8b18879c9384abfee5d0b685b00c03b3cf",
                  allow_patterns=["data/*.parquet"], local_dir="raw/cfpb_mouwiya", max_workers=8)
run = subprocess.run([sys.executable, "scripts/build_dataset.py", "--data", "raw/cfpb_mouwiya/data",
                      "--out", "data", "--since", "2023-08-25", "--until", "2026-06-17",
                      "--product-map", "data/product_map.json"], capture_output=True, text=True)
print(run.stdout[-1500:], run.stderr[-1500:])
run.check_returncode()

EXPECTED_IDS = {
    "real_train.jsonl": "b6b82fd9bb7810fcb11cf4d7acb7e14409c7957256fb800630d96b0c5313bfd1",
    "real_val.jsonl": "6059b064129d1a7914f4b0f99014b409be0065a6acb3be2ae58734ed937a21c6",
    "real_test.jsonl": "d92f9e68d6d6410fa99856846b1e1008d7daa5f9d30d9b9fdbecfe1020641fd9",
}
for name, want in EXPECTED_IDS.items():
    with open(os.path.join("data", name), encoding="utf-8") as f:
        ids = sorted(json.loads(l)["complaint_id"] for l in f)
    got = hashlib.sha256("\n".join(ids).encode()).hexdigest()
    assert got == want, f"{name}: complaint IDs differ from the frozen split"
print("OK: the complaint IDs match the team's frozen split.")

## 3. Run the second retriever

In [ ]:
!python -u rag/retrieve_v2.py

## 4. Download the result

In [ ]:
path = "data/passages_v2.jsonl"
with open(path, "rb") as f:
    print("rows:", sum(1 for _ in open(path, encoding="utf-8")),
          "| sha256:", hashlib.sha256(f.read()).hexdigest())
try:
    from google.colab import files
    files.download(path)
except ImportError:
    print("file is at", os.path.abspath(path))